# Main in-distribution results

* **Table 1** (`tab:overall_results`): averages over the 36 model/dataset
  combinations and calibration error (`MAIN_METRIC` in paper_utils.py) by model family
  and task type, with Wilcoxon stars.
* **Table 6** (`tab:all_acc`): accuracy of every model/dataset combination.
* **Appendix figures**: per-model (`main_results_5_*`) and per-dataset
  (`main_results_6_*`) results, and per-model / per-dataset reliability
  diagrams (`main_results_7_*`, `main_results_8_*`).

Inputs: `results/main.csv`, `results/main_blackbox.csv`, `results/basc_tp.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from paper_utils import (DATASETS_MAP, FIGURES_DIR, MAIN_METRIC, METHOD_COLOR, METHOD_LABEL, RESULTS_DIR,
                         TABLES_DIR, pooled_reliability, promote_semantic)

plt.style.use('seaborn-v0_8')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
INK = plt.rcParams['axes.labelcolor']
GRID = plt.rcParams.get('grid.color', '#cccccc')

def style_axes(ax, xlabel=None, ylabel=None):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.grid(True, color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    if xlabel is not None: ax.set_xlabel(xlabel, color=INK)
    if ylabel is not None: ax.set_ylabel(ylabel, color=INK)
    return ax

In [ ]:
# Methods reported in this notebook, in display order.
METHOD_ORDER = [
    'logprob',             # TP
    'ans_logprob',         # ATP
    'verbal_conf_stem6',   # VC
    'basc_k2',             # BASC-L (k=2)
    'basc_k5',             # BASC-L (k=5)
    'basc_tp',             # BASC-TP (Stage 2 on token probabilities)
    'basc_bb_k5',          # BASC-L-bb (Gemma embeddings)
    'oracle_sc_k100',      # TT-SC (100 test-time samples)
    'sup_logreg_iso_k1',   # supervised two-stage reference
]

df = promote_semantic(pd.read_csv(RESULTS_DIR / 'main.csv'))

bb = pd.read_csv(RESULTS_DIR / 'main_blackbox.csv')
bb = promote_semantic(bb[bb['method'].str.startswith('basc_')].copy())
bb['method'] = bb['method'].str.replace('basc_', 'basc_bb_', regex=False)

# BASC-TP: Stage 2 fit on all 400 calibration questions (800 decodes).
tp = pd.read_csv(RESULTS_DIR / 'basc_tp.csv')
tp = tp[tp['method'] == 'tp_iso_dcal'].assign(method='basc_tp')

df = pd.concat([df, bb, tp], ignore_index=True)
df['model_short'] = df['model'].str.split('/').str[-1]
df = df[df['method'].isin(METHOD_ORDER)].copy()
print(f'seeds={df["seed"].nunique()}, models={df["model"].nunique()}, '
      f'datasets={sorted(df["dataset"].unique())}, methods={df["method"].nunique()}')

In [ ]:
QWEN_MODELS = {'Qwen/Qwen3-0.6B', 'Qwen/Qwen3-1.7B', 'Qwen/Qwen3-4B-Thinking-2507',
               'Qwen/Qwen3-8B', 'Qwen/Qwen3-14B'}
NON_QWEN_MODELS = set(df['model'].unique()) - QWEN_MODELS
MATH_DATASETS = {'gsm8k', 'polymath'}
QA_DATASETS = {'sciq', 'trivia_qa', 'webq'}

METRICS = [
    ('ECE1',  'ECE$_1$ (lower is better)', False),
    ('ECE2',  'ECE$_2$ (lower is better)', False),
    ('Brier', 'Brier (lower is better)',   False),
    ('AUROC', 'AUROC (higher is better)',  True),
]

def combo_means(metric):
    """Seed-averaged metric per (method, model, dataset)."""
    return df.groupby(['method', 'model', 'dataset'])[metric].mean().reset_index()

def combo_mean_sem(metric, model_subset=None, dataset_subset=None):
    """Mean and SEM across (model, dataset) combinations."""
    per = combo_means(metric)
    if model_subset is not None:
        per = per[per['model'].isin(model_subset)]
    if dataset_subset is not None:
        per = per[per['dataset'].isin(dataset_subset)]
    out = per.groupby('method')[metric].agg(mean='mean', std='std', n='count').reset_index()
    out['sem'] = out['std'] / np.sqrt(out['n'].clip(lower=1))
    return out.set_index('method')

## Table 1
Left: averages over all combinations; right: `MAIN_METRIC` by model family and task type.
**Bold**: significantly better than each of TP, ATP and VC (paired two-sided
Wilcoxon test over the combinations in each column, p < 0.05).

In [ ]:
TABLE1_METHODS = METHOD_ORDER
STAR_BASELINES = ['logprob', 'ans_logprob', 'verbal_conf_stem6']
STAR_ALPHA = 0.05
GROUPS = [('Qwen', QWEN_MODELS, None), ('non-Qwen', NON_QWEN_MODELS, None),
          ('Math', None, MATH_DATASETS), ('QA', None, QA_DATASETS)]
LATEX_METRIC = {'ECE1': 'ECE$_1$', 'ECE2': 'ECE$_2$', 'Brier': 'Brier', 'AUROC': 'AUROC'}
MIDRULE_BEFORE = {'basc_bb_k5', 'oracle_sc_k100'}

def _combo_wide(metric, model_sub, ds_sub):
    per = combo_means(metric)
    if model_sub is not None:
        per = per[per['model'].isin(model_sub)]
    if ds_sub is not None:
        per = per[per['dataset'].isin(ds_sub)]
    return per.pivot_table(index=['model', 'dataset'], columns='method', values=metric)

def _beats_all_baselines(wide, method, higher_better):
    for b in STAR_BASELINES:
        pair = wide[[method, b]].dropna()
        diff = pair[method] - pair[b]
        better = diff.mean() > 0 if higher_better else diff.mean() < 0
        if not better or stats.wilcoxon(pair[method], pair[b]).pvalue >= STAR_ALPHA:
            return False
    return True

def _starred_col(methods, metric, model_sub=None, ds_sub=None):
    higher = {m: hb for m, _, hb in METRICS}[metric]
    wide = _combo_wide(metric, model_sub, ds_sub)
    means = wide.reindex(columns=methods).mean()
    col = pd.Series([f'{means[m]:.3f}' + ('*' if m.startswith('basc_') and
                                          _beats_all_baselines(wide, m, higher) else '')
                     for m in methods], index=methods)
    return col, len(wide)

ALL_METRICS = ('ECE1', 'ECE2', 'Brier', 'AUROC') if MAIN_METRIC == 'ECE2' else ('ECE1', 'Brier', 'AUROC')

def table1(methods):
    cols = {}
    for metric in ALL_METRICS:
        cols[('All', metric)], n = _starred_col(methods, metric)
    print(f'All: {n} (model, dataset) combinations')
    for name, model_sub, ds_sub in GROUPS:
        cols[(MAIN_METRIC, name)], n = _starred_col(methods, MAIN_METRIC, model_sub, ds_sub)
        print(f'{name}: {n} combinations')
    tbl = pd.DataFrame(cols, index=methods)
    tbl.index = [METHOD_LABEL[m] for m in methods]
    tbl.columns = pd.MultiIndex.from_tuples(tbl.columns)
    return tbl

def table1_latex(tbl, methods):
    n_all = sum(g == 'All' for g, _ in tbl.columns)
    n_grp = len(tbl.columns) - n_all
    lines = [rf' & \multicolumn{{{n_all}}}{{c}}{{All}} & \multicolumn{{{n_grp}}}{{c}}{{{LATEX_METRIC[MAIN_METRIC]}}} \\',
             ' & ' + ' & '.join(LATEX_METRIC[m] if g == 'All' else m for g, m in tbl.columns) + r' \\',
             r'\midrule']
    for m, (label, row) in zip(methods, tbl.iterrows()):
        if m in MIDRULE_BEFORE:
            lines.append(r'\midrule')
        cells = [rf'\textbf{{{v[:-1]}}}' if v.endswith('*') else v for v in row]
        lines.append(label + ' & ' + ' & '.join(cells) + r' \\')
    return '\n'.join(lines)

tbl1 = table1(TABLE1_METHODS)
tex1 = table1_latex(tbl1, TABLE1_METHODS)
(TABLES_DIR / 'table1_main_results.tex').write_text(tex1 + '\n')
print(tex1)
tbl1

## Table 6: accuracy per model and dataset

In [ ]:
MODEL_ORDER = [
    'Qwen3-0.6B', 'Qwen3-1.7B', 'Qwen3-4B-Thinking-2507', 'Qwen3-8B', 'Qwen3-14B',
    'OpenReasoning-Nemotron-7B', 'Nemotron-Cascade-8B-Thinking',
    'DeepSeek-R1-Distill-Qwen-7B', 'DeepSeek-R1-Distill-Llama-8B',
]

def accuracy_table():
    per_seed = df.drop_duplicates(subset=['model', 'dataset', 'seed'])[
        ['model', 'model_short', 'dataset', 'seed', 'accuracy']]
    tbl = per_seed.groupby(['model_short', 'dataset'])['accuracy'].mean().unstack('dataset')
    tbl = tbl.reindex(index=[m for m in MODEL_ORDER if m in tbl.index], columns=sorted(tbl.columns))
    tbl['mean'] = tbl.mean(axis=1)
    tbl.loc['mean'] = tbl.mean(axis=0)
    return tbl.rename(columns=lambda c: DATASETS_MAP.get(c, c))

tbl4 = accuracy_table()
tbl4_disp = tbl4.map(lambda v: f'{v:.3f}' if pd.notna(v) else '—')
tex6 = tbl4_disp.to_latex()
(TABLES_DIR / 'table6_accuracy.tex').write_text(tex6)
print(tex6)
tbl4_disp

## Appendix: per-model and per-dataset results

In [ ]:
PER_GROUP_METHODS = ['logprob', 'ans_logprob', 'verbal_conf_stem6', 'basc_k2', 'basc_k5',
                     'basc_tp', 'oracle_sc_k100']

def _bar_panel(ax, methods, means, sems, xlabel):
    y = np.arange(len(methods))
    ax.barh(y, means, xerr=sems, color=[METHOD_COLOR[m] for m in methods], edgecolor='white',
            linewidth=1.5, error_kw={'capsize': 3, 'lw': 1, 'alpha': 0.5})
    span = float(np.nanmax(means) - np.nanmin(means))
    pad = max(span * 0.05, float(np.nanmax(means)) * 0.05)
    for v, yi in zip(means, y):
        ax.text(v + pad, yi, f'{v:.3f}', va='center', ha='left', fontsize=10, color=INK)
    style_axes(ax, xlabel=xlabel)
    ax.set_xlim(min(0.0, float(np.nanmin(means)) - span * 0.1), float(np.nanmax(means)) + span * 0.3)

def plot_multi_metric_bars(methods, model_subset=None, dataset_subset=None, title=None, savepath=None):
    fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
    for ax, (metric, xlabel, _) in zip(axes, METRICS):
        agg = combo_mean_sem(metric, model_subset, dataset_subset).reindex(methods)
        _bar_panel(ax, methods, agg['mean'].values, agg['sem'].values, xlabel=xlabel)
    axes[0].set_yticks(np.arange(len(methods)))
    axes[0].set_yticklabels([METHOD_LABEL[m] for m in methods], color=INK)
    axes[0].invert_yaxis()
    if title:
        fig.suptitle(title, fontsize=15, color=INK, y=0.95)
    plt.tight_layout()
    if savepath:
        fig.savefig(savepath)
    plt.show()

for model in sorted(df['model'].unique()):
    model_short = model.split('/')[-1]
    plot_multi_metric_bars(PER_GROUP_METHODS, model_subset={model}, title=model_short,
                           savepath=FIGURES_DIR / f'main_results_5_{model_short}.png')

In [ ]:
for ds in sorted(df['dataset'].unique()):
    plot_multi_metric_bars(PER_GROUP_METHODS, dataset_subset={ds}, title=DATASETS_MAP[ds],
                           savepath=FIGURES_DIR / f'main_results_6_{ds}.png')

## Appendix: reliability diagrams per model and per dataset

In [ ]:
RELIABILITY_METHODS = ['logprob', 'verbal_conf_stem6', 'basc_k2', 'oracle_sc_k100']

def _reliability_panel(ax, methods, *, model_short=None, dataset=None):
    ax.plot([0, 1], [0, 1], color='k', lw=1.0, ls='--', zorder=1, label='perfect')
    for m in methods:
        out = pooled_reliability(df, m, model_short=model_short, dataset=dataset)
        if out is None:
            continue
        counts, mp, mc = out
        ok = counts > 0
        is_basc = 'basc' in m
        ax.plot(mp[ok], mc[ok], color=METHOD_COLOR[m], lw=1.8,
                marker='*' if is_basc else 'o', markersize=11 if is_basc else 5,
                markeredgecolor='white', markeredgewidth=0.8, label=METHOD_LABEL[m], zorder=3)
    style_axes(ax)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

fig, axes = plt.subplots(3, 3, figsize=(13, 8), sharex=True, sharey=True)
for ax, model_short in zip(axes.flat, MODEL_ORDER):
    _reliability_panel(ax, RELIABILITY_METHODS, model_short=model_short)
    ax.set_title(model_short, color=INK, fontsize=12)
for ax in axes[-1, :]:
    ax.set_xlabel('Predicted confidence', color=INK, fontsize=11)
for ax in axes[:, 0]:
    ax.set_ylabel('Empirical accuracy', color=INK, fontsize=11)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(handles), frameon=False,
           labelcolor=INK, fontsize=12, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Per-model reliability diagrams', color=INK, fontsize=15, y=0.97)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'main_results_7_reliability_per_model.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
RELIABILITY_DATASETS = ['gsm8k', 'polymath', 'sciq', 'trivia_qa', 'webq']
fig, axes = plt.subplots(1, len(RELIABILITY_DATASETS), figsize=(3.5 * len(RELIABILITY_DATASETS), 3.6),
                         sharey=True)
for ax, ds in zip(axes, RELIABILITY_DATASETS):
    _reliability_panel(ax, RELIABILITY_METHODS, dataset=ds)
    ax.set_title(DATASETS_MAP.get(ds, ds), color=INK, fontsize=13)
for ax in axes:
    ax.set_xlabel('Predicted confidence', color=INK, fontsize=11)
axes[0].set_ylabel('Empirical accuracy', color=INK, fontsize=11)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(handles), frameon=False,
           labelcolor=INK, fontsize=12, bbox_to_anchor=(0.5, -0.08))
fig.suptitle('Per-dataset reliability diagrams', color=INK, fontsize=15, y=0.97)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'main_results_8_reliability_per_dataset.png', dpi=300, bbox_inches='tight')
plt.show()